In [1]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

In [2]:
# Hyperparameters 
learning_rate = 0.0003
gamma = 0.98

In [3]:
from binance import Client
client = Client()
klines = np.array(client.futures_historical_klines("ETHUSDT", Client.KLINE_INTERVAL_1HOUR, "1 Jan, 2015"))
sample = pd.DataFrame(klines.reshape(-1, 12), dtype=float, columns=['datetime',
                                                                   'open',
                                                                   'high',
                                                                   'low',
                                                                   'close',
                                                                   'volume',
                                                                   'close time',
                                                                   'quote asset volume, number of trades',
                                                                   'number of trades',
                                                                   'taker buy base asset volume',
                                                                   'taker buy quote asset volume',
                                                                   'ignore'])
sample['datetime'] = pd.to_datetime(sample['datetime'], unit='ms')
sample.set_index('datetime', inplace=True)
sample = sample[['open', 'high', 'low', 'close', 'volume']]

In [4]:
# make states
# first make a band for 7 hours, 16 hours, 30 hours, 200 hours, 1500 hours
sample['p7-center'] = sample['close'].rolling(7).mean()
sample['p7-upper'] = sample['p7-center'] + 2.*sample['close'].rolling(7).std()
sample['p7-lower'] = sample['p7-center'] - 2.*sample['close'].rolling(7).std()
sample['p7-vcenter'] = sample['volume'].rolling(7).mean()
sample['p7-vupper'] = sample['p7-center'] + 2.*sample['volume'].rolling(7).std()
sample['p7-vlower'] = sample['p7-center'] - 2.*sample['volume'].rolling(7).std()

sample['p16-center'] = sample['close'].rolling(16).mean()
sample['p16-upper'] = sample['p16-center'] + 2.*sample['close'].rolling(16).std()
sample['p16-lower'] = sample['p16-center'] - 2.*sample['close'].rolling(16).std()
sample['p16-vcenter'] = sample['volume'].rolling(16).mean()
sample['p16-vupper'] = sample['p16-center'] + 2.*sample['volume'].rolling(16).std()
sample['p16-vlower'] = sample['p16-center'] - 2.*sample['volume'].rolling(16).std()

sample['p30-center'] = sample['close'].rolling(30).mean()
sample['p30-upper'] = sample['p30-center'] + 2.*sample['close'].rolling(30).std()
sample['p30-lower'] = sample['p30-center'] - 2.*sample['close'].rolling(30).std()
sample['p30-vcenter'] = sample['volume'].rolling(30).mean()
sample['p30-vupper'] = sample['p30-center'] + 2.*sample['volume'].rolling(30).std()
sample['p30-vlower'] = sample['p30-center'] - 2.*sample['volume'].rolling(30).std()

sample['p200-center'] = sample['close'].rolling(200).mean()
sample['p200-upper'] = sample['p200-center'] + 2.*sample['close'].rolling(200).std()
sample['p200-lower'] = sample['p200-center'] - 2.*sample['close'].rolling(200).std()
sample['p200-vcenter'] = sample['volume'].rolling(200).mean()
sample['p200-vupper'] = sample['p200-center'] + 2.*sample['volume'].rolling(200).std()
sample['p200-vlower'] = sample['p200-center'] - 2.*sample['volume'].rolling(200).std()

sample['p1500-center'] = sample['close'].rolling(7).mean()
sample['p1500-upper'] = sample['p1500-center'] + 2.*sample['close'].rolling(1500).std()
sample['p1500-lower'] = sample['p1500-center'] - 2.*sample['close'].rolling(1500).std()
sample['p1500-vcenter'] = sample['volume'].rolling(7).mean()
sample['p1500-vupper'] = sample['p1500-center'] + 2.*sample['volume'].rolling(1500).std()
sample['p1500-vlower'] = sample['p1500-center'] - 2.*sample['volume'].rolling(1500).std()

In [5]:
sample['p7-open'] = (sample['open'] - sample['p7-lower']) / (sample['p7-upper'] - sample['p7-lower'])
sample['p7-high'] = (sample['high'] - sample['p7-lower']) / (sample['p7-upper'] - sample['p7-lower'])
sample['p7-low'] = (sample['low'] - sample['p7-lower']) / (sample['p7-upper'] - sample['p7-lower'])
sample['p7-close'] = (sample['close'] - sample['p7-lower']) / (sample['p7-upper'] - sample['p7-lower'])
sample['p7-volume'] = (sample['volume'] - sample['p7-vlower']) / (sample['p7-vupper'] - sample['p7-vlower'])

sample['p16-open'] = (sample['open'] - sample['p16-lower']) / (sample['p16-upper'] - sample['p16-lower'])
sample['p16-high'] = (sample['high'] - sample['p16-lower']) / (sample['p16-upper'] - sample['p16-lower'])
sample['p16-low'] = (sample['low'] - sample['p16-lower']) / (sample['p16-upper'] - sample['p16-lower'])
sample['p16-close'] = (sample['close'] - sample['p16-lower']) / (sample['p16-upper'] - sample['p16-lower'])
sample['p16-volume'] = (sample['volume'] - sample['p16-vlower']) / (sample['p16-vupper'] - sample['p16-vlower'])

sample['p30-open'] = (sample['open'] - sample['p30-lower']) / (sample['p30-upper'] - sample['p30-lower'])
sample['p30-high'] = (sample['high'] - sample['p30-lower']) / (sample['p30-upper'] - sample['p30-lower'])
sample['p30-low'] = (sample['low'] - sample['p30-lower']) / (sample['p30-upper'] - sample['p30-lower'])
sample['p30-close'] = (sample['close'] - sample['p30-lower']) / (sample['p30-upper'] - sample['p30-lower'])
sample['p30-volume'] = (sample['volume'] - sample['p30-vlower']) / (sample['p30-vupper'] - sample['p30-vlower'])

sample['p200-open'] = (sample['open'] - sample['p200-lower']) / (sample['p200-upper'] - sample['p200-lower'])
sample['p200-high'] = (sample['high'] - sample['p200-lower']) / (sample['p200-upper'] - sample['p200-lower'])
sample['p200-low'] = (sample['low'] - sample['p200-lower']) / (sample['p200-upper'] - sample['p200-lower'])
sample['p200-close'] = (sample['close'] - sample['p200-lower']) / (sample['p200-upper'] - sample['p200-lower'])
sample['p200-volume'] = (sample['volume'] - sample['p200-vlower']) / (sample['p200-vupper'] - sample['p200-vlower'])

sample['p1500-open'] = (sample['open'] - sample['p1500-lower']) / (sample['p1500-upper'] - sample['p1500-lower'])
sample['p1500-high'] = (sample['high'] - sample['p1500-lower']) / (sample['p1500-upper'] - sample['p1500-lower'])
sample['p1500-low'] = (sample['low'] - sample['p1500-lower']) / (sample['p1500-upper'] - sample['p1500-lower'])
sample['p1500-close'] = (sample['close'] - sample['p1500-lower']) / (sample['p1500-upper'] - sample['p1500-lower'])
sample['p1500-volume'] = (sample['volume'] - sample['p1500-vlower']) / (sample['p1500-vupper'] - sample['p1500-vlower'])

sample.dropna(inplace=True)

In [6]:
cut1 = int(len(sample)*0.8)
cut2 = int(len(sample)*0.85)
train_sample, val_sample, test_sample = sample.iloc[:cut1], sample.iloc[cut1:cut2], sample.iloc[cut2:]

In [7]:
### Evaluated buy and hold strategy for train, validation, test samples
book = train_sample[['close']].copy()
book['number'] = book.index.map(mdates.date2num)
book['reward'] = 1. + book['close'].pct_change()
book['total_reward'] = book['reward'].cumprod()

CAGR = book['total_reward'].iloc[-1]**(365*24/len(book.index)) - 1.
historical_max = book['total_reward'].cummax()
daily_drawdown = book['total_reward']/historical_max - 1.
historical_dd = daily_drawdown.cummin()
MDD = historical_dd.min()
VOL = np.std(book['reward'])*np.sqrt(365.*24)
Sharpe = (np.mean(book['reward'])/np.std(book['reward'])*np.sqrt(365.*24))

print("==== BUY AND HOLD ====")
print(f"Accumulated Returns: {(book['total_reward'].iloc[-1]-1.)*100:.2f} %")
print(f"CAGR: {CAGR*100:.2f} %")
print(f"MDD: {MDD*100:.2f} %")
print(f"VOL: {VOL*100:.2f}%")
print(f"Sharpe: {Sharpe*100:.2f} %")


==== BUY AND HOLD ====
Accumulated Returns: 2011.13 %
CAGR: 504.41 %
MDD: -66.59 %
VOL: 106.10%
Sharpe: 825849.45 %


In [8]:
### Evaluated buy and hold strategy for train, validation, test samples
book = val_sample[['close']].copy()
book['number'] = book.index.map(mdates.date2num)
book['reward'] = 1. + book['close'].pct_change()
book['total_reward'] = book['reward'].cumprod()

CAGR = book['total_reward'].iloc[-1]**(365*24/len(book.index)) - 1.
historical_max = book['total_reward'].cummax()
daily_drawdown = book['total_reward']/historical_max - 1.
historical_dd = daily_drawdown.cummin()
MDD = historical_dd.min()
VOL = np.std(book['reward'])*np.sqrt(365.*24)
Sharpe = (np.mean(book['reward'])/np.std(book['reward'])*np.sqrt(365.*24))

print("==== BUY AND HOLD ====")
print(f"Accumulated Returns: {(book['total_reward'].iloc[-1]-1.)*100:.2f} %")
print(f"CAGR: {CAGR*100:.2f} %")
print(f"MDD: {MDD*100:.2f} %")
print(f"VOL: {VOL*100:.2f}%")
print(f"Sharpe: {Sharpe*100:.2f} %")

==== BUY AND HOLD ====
Accumulated Returns: 19.67 %
CAGR: 444.73 %
MDD: -11.13 %
VOL: 64.40%
Sharpe: 1360535.38 %


In [9]:
### Evaluated buy and hold strategy for train, validation, test samples
book = test_sample[['close']].copy()
book['number'] = book.index.map(mdates.date2num)
book['reward'] = 1. + book['close'].pct_change()
book['total_reward'] = book['reward'].cumprod()

CAGR = book['total_reward'].iloc[-1]**(365*24/len(book.index)) - 1.
historical_max = book['total_reward'].cummax()
daily_drawdown = book['total_reward']/historical_max - 1.
historical_dd = daily_drawdown.cummin()
MDD = historical_dd.min()
VOL = np.std(book['reward'])*np.sqrt(365.*24)
Sharpe = (np.mean(book['reward'])/np.std(book['reward'])*np.sqrt(365.*24))

print("==== BUY AND HOLD ====")
print(f"Accumulated Returns: {(book['total_reward'].iloc[-1]-1.)*100:.2f} %")
print(f"CAGR: {CAGR*100:.2f} %")
print(f"MDD: {MDD*100:.2f} %")
print(f"VOL: {VOL*100:.2f}%")
print(f"Sharpe: {Sharpe*100:.2f} %")

==== BUY AND HOLD ====
Accumulated Returns: -40.46 %
CAGR: -80.42 %
MDD: -54.22 %
VOL: 85.27%
Sharpe: 1027157.40 %


In [14]:
class Environment():
    def __init__(self, sample):
        self.sample = sample[
                        ['open', 'high', 'low', 'close', 'volume',
                         'p7-open', 'p7-high', 'p7-low', 'p7-close', 'p7-volume',
                         'p16-open', 'p16-high', 'p16-low', 'p16-close', 'p16-volume',
                         'p30-open', 'p30-high', 'p30-low', 'p30-close', 'p30-volume',
                         'p200-open', 'p200-high', 'p200-low', 'p200-close', 'p200-volume',
                         'p1500-open', 'p1500-high', 'p1500-low', 'p1500-close', 'p1500-volume']
                        ].copy()
        self.episode = None
        self.epi_idx = 0
        
    def make_episode(self, start_idx=-1):
        # check start index
        if start_idx > len(self.sample) - 2000:
            print(f"start_idx should be smaller than {len(self.sample)-2000}")
            raise(ValueError)
        if start_idx == -1:
            start_idx = random.randint(0, len(self.sample)-2000)
            
        # make an episode
        self.episode = self.sample[start_idx:start_idx+2000]
        self.epi_idx = 0
        first_state = self.episode.iloc[self.epi_idx].to_numpy()[5:]
        return first_state
    
    def step(self, action):
        if action == 0: action = "short"
        elif action == 1: action = "hold"
        elif action == 2: action = "long"
        else: exit()
        
        this_close = self.episode.iloc[self.epi_idx].to_numpy()[3]
        
        self.epi_idx += 1
        
        # is done?
        done = True if self.epi_idx == 2000-1 else False
        
        # make state
        state = self.episode.iloc[self.epi_idx].to_numpy()[5:]
        
        # make reward
        next_high = self.episode.iloc[self.epi_idx].to_numpy()[1]
        next_low = self.episode.iloc[self.epi_idx].to_numpy()[2]
        reward = np.log(random.uniform(next_low, next_high)/this_close)
        
        if action == "long":    reward -= 0.002
        elif action == "short": reward = -1*reward - 0.002
        else:                   reward = 0. 
        
        # index and ohlcv
        info = (self.epi_idx, self.episode.iloc[self.epi_idx].to_numpy()[:5])
        
        return (state, reward, done, info)

In [16]:
class Policy(nn.Module):
    def __init__(self):
        super(Policy, self).__init__()
        self.data = []
        
        # nonliear aggregation of ohlcv
        self.agg1 = nn.Linear(5, 5, bias=True)
        self.agg2 = nn.Linear(5, 5, bias=True)
        self.agg3 = nn.Linear(5, 5, bias=True)
        self.agg4 = nn.Linear(5, 5, bias=True)
        self.agg5 = nn.Linear(5, 5, bias=True)
        
        self.fc1 = nn.Linear(25, 128, bias=True)
        self.fc2 = nn.Linear(128, 128, bias=True)
        self.fc3 = nn.Linear(128, 3)
        self.dropout = nn.Dropout(p=0.2)
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        
        # history
        self.episodes = 0
        self.scores = list()
        self.loss = list()
        
    def forward(self, x):
        agg1 = F.relu(self.agg1(x[0:5]))
        agg2 = F.relu(self.agg2(x[5:10]))
        agg3 = F.relu(self.agg3(x[10:15]))
        agg4 = F.relu(self.agg4(x[15:20]))
        agg5 = F.relu(self.agg5(x[20:25]))
        x = torch.concat([agg1, agg2, agg3, agg4, agg5])
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.softmax(self.fc3(x), dim=0)
        return x
    
    def put_data(self, item):
        self.data.append(item)
    
    def train_net(self):
        R = 0.
        self.optimizer.zero_grad()
        for r, prob in self.data[::-1]:
            R = r + gamma*R
            loss = -R*torch.log(prob)
            loss.backward()
        self.optimizer.step()
        self.data = []
        
        return loss.detach().numpy()

In [12]:
class History():
    def __init__(self):
        self.episodes = 0
        self.score = []
        self.loss = []
        self.evaluation = {}
        self.evaluation['index'] = []
        self.evaluation['CAGR'] = []
        self.evaluation['MDD'] = []
        self.evaluation['VOL'] = []
        self.evaluation['Sharpe'] = []
    
    def update(self, score, loss):
        self.episodes += 1
        self.score.append(score)
        self.loss.append(loss)
        
    def save_evaluation(self, index, args):
        CAGR, MDD, VOL, Sharpe = args
        self.evaluation['index'].append(index)
        self.evaluation['CAGR'].append(CAGR)
        self.evaluation['MDD'].append(MDD)
        self.evaluation['VOL'].append(VOL)
        self.evaluation['Sharpe'].append(Sharpe)
    
    def plot(self):
        plt.figure(figsize=(15, 8))
        plt.subplot(2, 1, 1)
        plt.plot(range(self.episodes), self.score, 'r--', label='score')
        plt.legend(loc='best')
        
        plt.subplot(2, 1, 2)
        plt.plot(range(self.episodes), self.loss, 'b--', label='loss')
        plt.legend(loc='best')
        
        plt.show()

In [17]:
env = Environment(train_sample)
pi = Policy()
train_history = History()
val_history = History()
acc_score = 0.
print_interval = 20
estimate_interval = 1000

for n_epi in range(10000):
    score= 0.
    state = env.make_episode()
    done = False
    
    while not done:
        prob = pi(torch.from_numpy(state).float())
        m = Categorical(prob)
        action = m.sample()
        next_state, reward, done, info = env.step(action.item())
        pi.put_data((reward, prob[action]))
        state = next_state
        score += reward
        
    pi.train()
    loss = pi.train_net()
    train_history.update(score, loss)
    
    acc_score += score
    if n_epi%print_interval == 0 and n_epi != 0:
        print(f"[EPISODE {n_epi}] avg return: {(np.exp(acc_score/print_interval)-1)*100:.2f}%")
        acc_score = 0.

train_history.plot()

[EPISODE 20] avg return: -94.36%
[EPISODE 40] avg return: -91.70%
[EPISODE 60] avg return: -90.46%
[EPISODE 80] avg return: -86.78%
[EPISODE 100] avg return: -79.50%
[EPISODE 120] avg return: -63.59%
[EPISODE 140] avg return: -40.25%
[EPISODE 160] avg return: -23.68%
[EPISODE 180] avg return: -14.11%
[EPISODE 200] avg return: -5.12%
[EPISODE 220] avg return: -6.40%
[EPISODE 240] avg return: -4.10%
[EPISODE 260] avg return: -3.67%
[EPISODE 280] avg return: -1.88%
[EPISODE 300] avg return: -2.84%
[EPISODE 320] avg return: -1.04%
[EPISODE 340] avg return: 0.76%
[EPISODE 360] avg return: -1.08%
[EPISODE 380] avg return: -0.65%
[EPISODE 400] avg return: -1.23%
